# Energy use vs CO₂ (country-year)
Uses only the Python standard library so it runs offline with no packages.

1. Keep rows whose `iso_code` is a 3-letter country (drop `OWID_WRL`, `OWID_EUR`, blank ISO, `World`).
2. Inner-join energy to CO₂ on `iso_code` + `year` (one-to-one).
3. Energy is TWh; `co2` is million tonnes — do not treat them as the same unit.

Replace sample paths with the OWID energy and CO₂ CSVs after download. Result grain is **country-year**.


In [ ]:
import csv
from pathlib import Path

root = Path(".")
DROP = {"OWID_WRL", "OWID_EUR", "OWID_AFR", "WLD", "EUU"}
NAME_DROP = {"world", "africa", "asia", "europe", "european union"}

def keep(iso, name=""):
    code = (iso or "").strip().upper()
    if len(code) != 3 or code in DROP or code.startswith("OWID_"):
        return False
    return str(name).strip().lower() not in NAME_DROP

def load(path, value):
    out = {}
    with (root / path).open() as fh:
        for row in csv.DictReader(fh):
            if not keep(row["iso_code"], row["country"]):
                continue
            key = (row["iso_code"].upper(), int(row["year"]))
            if key in out:
                raise SystemExit(f"duplicate {key} in {path}")
            out[key] = float(row[value])
    return out

energy = load("data/samples/owid-energy-sample.csv", "primary_energy_consumption")
co2 = load("data/samples/owid-co2-sample.csv", "co2")
assert ("OWID_WRL", 2020) not in energy
joined = []
for key, twh in sorted(energy.items()):
    if key not in co2:
        continue
    joined.append((key[0], key[1], twh, co2[key]))
keys = [r[:2] for r in joined]
assert len(keys) == len(set(keys)), "join is not one-to-one"
assert joined, "no overlapping country-years"
print("iso year energy_twh co2_mt")
for row in joined:
    print(*row)
print(f"{len(joined)} country-year rows after dropping aggregates")
